# Random init jitter — does Gaussian noise help or hurt bound/classic transfer?

After `synthetic_compare.ipynb` (concept-bound init) and `direction_aware_cardinality.ipynb` (direction tags for the $\delta$ count channel), we ask whether **adding random Gaussian noise to every initialized row** changes what a linear probe can read at epoch 0 and what survives protected finetuning.

The noise hook already lives in `protograph_init(..., noise=σ)` in `scripts/_synthetic_compare.py`: after class-mean / bound vectors are copied into `wv` and `syn1neg`, each row gets `v ← v + 𝒩(0, σ²)` with a fixed RNG seed (independent of the walk seed).

## Factorial

| axis | levels |
|---|---|
| protograph | P1, P2, P3 |
| init | `classic` (own-class mean), `bound` (concept-bound superposition) |
| direction-aware cardinality | `off` (`shared` incoming code — direction-blind), `on` (`rolled` — current default) |
| jitter σ | 0, 0.05, 0.1, 0.2, 0.5 |

`classic` ignores the direction axis (no bound construction). For `bound`, `off`/`on` swap the incoming-edge tag in both the $\gamma$ binding and $\delta$ count terms — same ablation as `direction_aware_cardinality.ipynb`.

## Experiment ladder

1. **Init-space probe** — LogReg on jittered init vectors, no finetuning. Is separability destroyed immediately?
2. **Full pipeline** — pretrain → jittered init → 5 protected finetune epochs. Does noise let `*_classic` escape the ~0.5 plateau, or erode `*_bound`?
3. **σ sweep curves** — init vs final accuracy as a function of σ, faceted by variant and direction setting.

Reference rows at σ=0 with direction `on` for `*_bound` should match cached `synthetic_compare` / `direction_aware_cardinality` numbers (within run-to-run variance).

In [ ]:
"""Setup: paths, helpers, configuration."""
import json
import pickle
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gensim.models.word2vec import LineSentence
from sklearn.metrics import accuracy_score

ROOT = Path("..").resolve()
SCRIPTS = ROOT / "scripts"
COMPARE_ROOT = ROOT / "notebooks" / "synthetic_compare"
OUT_ROOT = ROOT / "notebooks" / "random_jitter"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from _evaluate import load_labeled_txt, make_classifiers  # noqa: E402
from _maschine_init import stage1_vector_lookup  # noqa: E402
from _protograph_gen import iter_rdf_iris  # noqa: E402
from _synthetic_compare import (  # noqa: E402
    concept_bound_vectors,
    ensure_walks,
    load_materialized_types,
    make_eval_fn,
    new_skipgram_model,
    normalized_stage1_vectors,
    pretrain_protograph,
    protograph_init,
    train_with_eval,
    write_protographs,
)

# DLCC focus cases where bound init matters (tc07: qualified existence, tc09–tc12: cardinality)
TCS = ["tc07", "tc09", "tc10", "tc11", "tc12"]
PROTOS = ("p1", "p2", "p3")
NOISE_LEVELS = (0.0, 0.05, 0.1, 0.2, 0.5)
DIRECTION = dict(off="shared", on="rolled")

CFG = dict(
    dim=200,
    walks_per_entity=100,
    proto_walks_per_entity=200,
    depth=3,
    epochs=5,
    pretrain_epochs=5,
    finetune_alpha=0.0025,
    min_alpha=0.0001,
    target_norm=8.0,
    seed=42,
)


def tc_paths(tc: str) -> dict:
    tc_dir = ROOT / "v1" / "synthetic_ontology" / tc / "synthetic_ontology"
    return dict(
        ontology=tc_dir / "ontology.nt",
        graph=tc_dir / "graph.nt",
        train=tc_dir / "1000" / "train_test" / "train.txt",
        test=tc_dir / "1000" / "train_test" / "test.txt",
        cache=COMPARE_ROOT / tc,
        out=OUT_ROOT / tc,
    )


def inner_iri(token: str) -> str:
    return token[1:-1] if token.startswith("<") and token.endswith(">") else token


def labeled_splits(tc: str):
    p = tc_paths(tc)
    train_tokens, y_tr = load_labeled_txt(p["train"])
    test_tokens, y_te = load_labeled_txt(p["test"])
    return [inner_iri(t) for t in train_tokens], y_tr, [inner_iri(t) for t in test_tokens], y_te


for tc in TCS:
    for k, p in tc_paths(tc).items():
        if k in ("cache", "out"):
            continue
        assert p.is_file(), f"missing {k} for {tc}: {p}"
    tc_paths(tc)["out"].mkdir(parents=True, exist_ok=True)  # stage1_*.pkl cache

print(f"Output root: {OUT_ROOT}")
print(f"TCs: {', '.join(TCS)}")
print(f"Noise levels σ: {NOISE_LEVELS}")

## Helpers — directional bound vectors + jittered pipeline run

Reuses the direction-tag machinery from `direction_aware_cardinality.ipynb`. Cached P{1,2,3} pretrain codes live under each TC's output folder.

In [ ]:
def tag_code(rc: np.ndarray, tag: str) -> np.ndarray:
    if tag == "shared":
        return rc
    if tag == "rolled":
        return np.roll(rc, 1)
    raise ValueError(f"unknown direction tag {tag!r}")


def directional_bound_vectors(
    graph_nt: Path,
    ontology_nt: Path,
    stage1_vectors: dict[str, np.ndarray],
    *,
    direction_tag: str = "rolled",
    target_norm: float = 1.0,
) -> dict[str, np.ndarray]:
    types = load_materialized_types(ontology_nt)

    def unit(v: np.ndarray) -> np.ndarray:
        n = float(np.linalg.norm(v))
        return v / n if n > 0 else v

    codes: dict[str, np.ndarray] = {}

    def code_of(inner: str) -> np.ndarray | None:
        if inner in codes:
            return codes[inner]
        vec = stage1_vector_lookup(inner, stage1_vectors)
        if vec is None:
            return None
        u = unit(np.asarray(vec, dtype=np.float32))
        codes[inner] = u
        return u

    def class_mix(inners) -> np.ndarray | None:
        parts = [code_of(c) for c in inners]
        parts = [p for p in parts if p is not None]
        return unit(np.mean(parts, axis=0)) if parts else None

    acc: dict[str, np.ndarray] = {}

    def bump(ent: str, vec: np.ndarray, w: float = 1.0) -> None:
        if ent not in acc:
            acc[ent] = np.zeros_like(vec)
        acc[ent] += w * vec

    for ent, cs in types.items():
        mix = class_mix(cs)
        if mix is not None:
            bump(ent, mix)

    for s, r, o in iter_rdf_iris(graph_nt):
        rc = code_of(r)
        if rc is None:
            continue
        rc_inv = tag_code(rc, direction_tag)
        bump(s, rc)
        bump(o, rc_inv)
        for c in types.get(o, ()):
            cc = code_of(c)
            if cc is not None:
                bump(s, unit(rc * cc))
        for c in types.get(s, ()):
            cc = code_of(c)
            if cc is not None:
                bump(o, unit(rc_inv * cc))

    norms = [float(np.linalg.norm(v)) for v in acc.values()]
    mean_norm = float(np.mean(norms)) if norms else 1.0
    scale = target_norm / mean_norm if mean_norm > 0 else 1.0
    return {
        f"<{ent}>": (vec * scale).astype(np.float32)
        for ent, vec in acc.items()
        if float(np.linalg.norm(vec)) > 0
    }


def stage1_codes(tc: str, proto: str) -> tuple[dict[str, np.ndarray], float]:
    pkl = tc_paths(tc)["out"] / f"stage1_{proto}.pkl"
    if pkl.is_file():
        return pickle.loads(pkl.read_bytes())
    p = tc_paths(tc)
    p["cache"].mkdir(parents=True, exist_ok=True)
    proto_paths = write_protographs(p["ontology"], p["cache"])
    walks = ensure_walks(
        proto_paths[proto], p["cache"] / f"walks_{proto}.txt",
        walks_per_entity=CFG["proto_walks_per_entity"], depth=CFG["depth"],
        seed=CFG["seed"], ensure_triple_coverage=True,
    )
    pre = pretrain_protograph(
        walks, dim=CFG["dim"], epochs=CFG["pretrain_epochs"], seed=CFG["seed"]
    )
    stage1, used_norm = normalized_stage1_vectors(pre.wv, target_norm=CFG["target_norm"])
    pkl.write_bytes(pickle.dumps((stage1, used_norm)))
    return stage1, used_norm


def variant_key(proto: str, init: str, direction: str | None) -> str:
    if init == "classic":
        return f"{proto}_classic"
    tag = DIRECTION[direction]
    return f"{proto}_bound_{direction}"


def build_bound_vectors(tc: str, proto: str, direction: str) -> dict[str, np.ndarray]:
    p = tc_paths(tc)
    stage1, used_norm = stage1_codes(tc, proto)
    tag = DIRECTION[direction]
    if tag == "rolled":
        # identical to concept_bound_vectors (default implementation)
        return concept_bound_vectors(
            p["graph"], p["ontology"], stage1, target_norm=used_norm,
        )
    return directional_bound_vectors(
        p["graph"], p["ontology"], stage1, direction_tag=tag, target_norm=used_norm,
    )


def jitter_run(
    tc: str,
    proto: str,
    init: str,
    *,
    direction: str | None = None,
    noise: float = 0.0,
) -> dict:
    p = tc_paths(tc)
    stage1, used_norm = stage1_codes(tc, proto)
    inst_walks = ensure_walks(
        p["graph"],
        p["cache"] / f"walks_instance_w{CFG['walks_per_entity']}_d{CFG['depth']}.txt",
        walks_per_entity=CFG["walks_per_entity"], depth=CFG["depth"], seed=CFG["seed"],
    )
    bound_vectors = None
    if init == "bound":
        bound_vectors = build_bound_vectors(tc, proto, direction)

    model = new_skipgram_model(
        dim=CFG["dim"], alpha=CFG["finetune_alpha"], min_alpha=CFG["min_alpha"],
        seed=CFG["seed"], workers=16,
    )
    model.build_vocab(LineSentence(str(inst_walks)))
    init_stats = protograph_init(
        model, stage1, p["ontology"], strategy="all_init",
        target_norm=used_norm, bound_vectors=bound_vectors, noise=noise,
    )
    accs = train_with_eval(
        model, inst_walks, epochs=CFG["epochs"], alpha=CFG["finetune_alpha"],
        min_alpha=CFG["min_alpha"], eval_fn=make_eval_fn(p["train"], p["test"]),
    )
    return dict(
        tc=tc, proto=proto, init=init, direction=direction, noise=noise,
        variant=variant_key(proto, init, direction),
        accs=accs, init_acc=float(accs[0]), final_acc=float(accs[-1]),
        init_stats=init_stats,
    )


def probe_matrix(vectors: dict[str, np.ndarray], tokens: list[str], dim: int) -> np.ndarray:
    x = np.zeros((len(tokens), dim), dtype=np.float32)
    for i, tok in enumerate(tokens):
        v = vectors.get(f"<{tok}>")
        if v is not None:
            x[i] = v
    return x


def apply_jitter(vectors: dict[str, np.ndarray], noise: float, seed: int = 12345) -> dict[str, np.ndarray]:
    """Mirror protograph_init noise: same fixed RNG seed, applied to every row."""
    if noise <= 0:
        return vectors
    rng = np.random.default_rng(seed)
    return {
        tok: (v + rng.normal(0.0, noise, size=v.shape)).astype(np.float32)
        for tok, v in vectors.items()
    }


def init_vectors_for_variant(tc: str, proto: str, init: str, direction: str | None) -> dict[str, np.ndarray]:
    p = tc_paths(tc)
    stage1, used_norm = stage1_codes(tc, proto)
    if init == "bound":
        return build_bound_vectors(tc, proto, direction)
    # classic: class-mean init for labeled entities only (probe subset)
    from _maschine_init import (  # noqa: E402
        build_entity_to_class_tokens,
        load_maschine_entity_mapping,
        maschine_embedding_for_token,
        normalize_init_strategy,
    )
    parents, entity_types = load_maschine_entity_mapping(p["ontology"], quiet=True)
    ent_to_classes = build_entity_to_class_tokens(parents, entity_types)
    strategy = normalize_init_strategy("all_init")
    out = {}
    tr, _, te, _ = labeled_splits(tc)
    for ent in set(tr) | set(te):
        tok = f"<{ent}>"
        vec = maschine_embedding_for_token(
            tok, stage1, parents, ent_to_classes, strategy=strategy,
        )
        if vec is not None:
            v = np.asarray(vec, dtype=np.float32)
            n = float(np.linalg.norm(v))
            if n > 0:
                v = v * (used_norm / n)
            out[tok] = v
    return out


print("Helpers ready.")

## Part A — init-space probe with jitter

LogReg on jittered init vectors (no finetuning). Measures how quickly separability decays with σ. Cached in `probe_results.json`.

In [ ]:
PROBE_JSON = OUT_ROOT / "probe_results.json"
probe_results = json.loads(PROBE_JSON.read_text()) if PROBE_JSON.is_file() else {}
t0 = time.time()

for tc in TCS:
    tr, y_tr, te, y_te = labeled_splits(tc)
    clf = make_classifiers(max_iter=1000, seed=CFG["seed"])["LogReg"]
    for proto in PROTOS:
        for init in ("classic", "bound"):
            directions = (None,) if init == "classic" else ("off", "on")
            for direction in directions:
                base = init_vectors_for_variant(tc, proto, init, direction)
                for noise in NOISE_LEVELS:
                    key = f"{tc}|{variant_key(proto, init, direction)}|σ={noise}"
                    if key in probe_results:
                        continue
                    vecs = apply_jitter(base, noise)
                    x_tr = probe_matrix(vecs, tr, CFG["dim"])
                    x_te = probe_matrix(vecs, te, CFG["dim"])
                    clf.fit(x_tr, y_tr)
                    acc = float(accuracy_score(y_te, clf.predict(x_te)))
                    probe_results[key] = acc
                    print(f"[{time.time()-t0:6.1f}s] {key:40s} acc={acc:.3f}", flush=True)

PROBE_JSON.write_text(json.dumps(probe_results, indent=2) + "\n")

probe_rows = []
for key, acc in probe_results.items():
    tc, variant, sigma = key.split("|")
    sigma = float(sigma.split("=")[1])
    parts = variant.split("_")
    proto = parts[0]
    init = parts[1]
    direction = parts[2] if len(parts) > 2 else None
    probe_rows.append(dict(tc=tc, variant=variant, proto=proto, init=init,
                           direction=direction, noise=sigma, acc=acc))
probe_df = pd.DataFrame(probe_rows)

for init in ("classic", "bound"):
    sub = probe_df[probe_df.init == init]
    if init == "bound":
        for direction in ("off", "on"):
            s = sub[sub.direction == direction]
            table = s.pivot_table(index=["tc", "variant"], columns="noise", values="acc", aggfunc="first")
            print(f"\nInit probe — bound, direction {direction}")
            display(table.round(3))
    else:
        table = sub.pivot_table(index=["tc", "variant"], columns="noise", values="acc", aggfunc="first")
        print("\nInit probe — classic")
        display(table.round(3))

## Part B — full pipeline with jitter

Pretrain → jittered init → 5 protected finetune epochs. Cached in `pipeline_results.json`.

In [ ]:
PIPELINE_JSON = OUT_ROOT / "pipeline_results.json"
# Long run (~2 h for all TCs): `python notebooks/random_jitter/run_pipeline.py`
pipeline_results = json.loads(PIPELINE_JSON.read_text()) if PIPELINE_JSON.is_file() else {}
t0 = time.time()

for tc in TCS:
    for proto in PROTOS:
        for init in ("classic", "bound"):
            directions = (None,) if init == "classic" else ("off", "on")
            for direction in directions:
                for noise in NOISE_LEVELS:
                    key = f"{tc}|{variant_key(proto, init, direction)}|σ={noise}"
                    if key in pipeline_results:
                        continue
                    r = jitter_run(tc, proto, init, direction=direction, noise=noise)
                    pipeline_results[key] = r
                    accs = " ".join(f"{a:.3f}" for a in r["accs"])
                    print(f"[{time.time()-t0:6.1f}s] {key:40s} [{accs}]", flush=True)
                    PIPELINE_JSON.write_text(json.dumps(pipeline_results, indent=2) + "\n")

pipe_rows = []
for key, r in pipeline_results.items():
    pipe_rows.append(dict(
        tc=r["tc"], variant=r["variant"], proto=r["proto"], init=r["init"],
        direction=r["direction"], noise=r["noise"],
        init_acc=r["init_acc"], final_acc=r["final_acc"], accs=r["accs"],
    ))
pipe_df = pd.DataFrame(pipe_rows)

# Reference: σ=0 from synthetic_compare (bound on) and our own σ=0 runs
compare = json.loads((COMPARE_ROOT / "results.json").read_text())
ref_rows = []
for tc in TCS:
    for row in compare.get(tc, []):
        if row["variant"] in ("vanilla", "p1_bound", "p2_bound", "p3_bound",
                                "p1_classic", "p2_classic"):
            ref_rows.append(dict(
                tc=tc, variant=row["variant"], source="synthetic_compare",
                init_acc=row["accs"][0], final_acc=row["final_acc"],
            ))
ref_df = pd.DataFrame(ref_rows)
print("\nCached reference (σ=0, no extra jitter) from synthetic_compare.ipynb")
display(ref_df.pivot(index="tc", columns="variant", values="final_acc").round(3))

print("\nOur σ=0 pipeline (should match references for bound+on / classic)")
zero = pipe_df[pipe_df.noise == 0.0]
display(zero.pivot_table(index="tc", columns="variant", values="final_acc", aggfunc="first").round(3))

## Part C — σ sweep plots and deltas

How init and final accuracy move with σ, and jitter gain/loss vs σ=0 baseline.

In [ ]:
def plot_sigma_curves(df: pd.DataFrame, metric: str, title: str, fname: str) -> None:
    variants = sorted(df["variant"].unique())
    n = len(variants)
    ncol = 3
    nrow = (n + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3 * nrow), sharex=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, variant in zip(axes, variants):
        sub = df[df.variant == variant]
        for tc in TCS:
            s = sub[sub.tc == tc].sort_values("noise")
            ax.plot(s["noise"], s[metric], marker="o", label=tc, alpha=0.8)
        ax.axhline(0.5, color="gray", ls="--", lw=0.8, alpha=0.5)
        ax.set_title(variant, fontsize=9)
        ax.set_ylim(0.4, 1.02)
    for ax in axes[len(variants):]:
        ax.set_visible(False)
    axes[0].legend(fontsize=7, ncol=2)
    fig.supxlabel("jitter σ")
    fig.supylabel(metric)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    fig.savefig(OUT_ROOT / fname, dpi=120, bbox_inches="tight")
    plt.show()


plot_sigma_curves(pipe_df, "init_acc", "Init accuracy vs jitter σ (full pipeline epoch 0)", "init_vs_sigma.png")
plot_sigma_curves(pipe_df, "final_acc", "Final accuracy vs jitter σ (after 5 finetune epochs)", "final_vs_sigma.png")
plot_sigma_curves(probe_df, "acc", "Init probe accuracy vs jitter σ (no finetuning)", "probe_vs_sigma.png")

# Delta vs σ=0 baseline
baseline = pipe_df[pipe_df.noise == 0.0].set_index(["tc", "variant"])["final_acc"]
delta_rows = []
for _, row in pipe_df.iterrows():
    key = (row["tc"], row["variant"])
    if key in baseline.index:
        delta_rows.append(dict(
            tc=row["tc"], variant=row["variant"], noise=row["noise"],
            final_acc=row["final_acc"],
            delta=row["final_acc"] - baseline[key],
        ))
delta_df = pd.DataFrame(delta_rows)

print("\nMean final-acc delta vs σ=0 (negative = jitter hurt)")
delta_summary = (
    delta_df.groupby(["variant", "noise"])["delta"].mean()
    .unstack("noise").round(3)
)
display(delta_summary)

# Best σ per variant (mean over TCs)
best_rows = []
for variant in sorted(pipe_df.variant.unique()):
    sub = pipe_df[pipe_df.variant == variant]
    means = sub.groupby("noise")["final_acc"].mean()
    best_sigma = float(means.idxmax())
    best_rows.append(dict(
        variant=variant, best_sigma=best_sigma,
        mean_final=means[best_sigma], mean_init=sub.groupby("noise")["init_acc"].mean()[best_sigma],
        sigma0_final=means.get(0.0, np.nan),
    ))
best_df = pd.DataFrame(best_rows).set_index("variant")
print("\nBest mean final accuracy per variant")
display(best_df.round(3))

### Saved σ-sweep plots

Re-run Part C to regenerate, or display the cached figures written during the experiment run:

In [ ]:
from IPython.display import Image, display

for fname in ("probe_vs_sigma.png", "init_vs_sigma.png", "final_vs_sigma.png"):
    path = OUT_ROOT / fname
    if path.is_file():
        print(fname)
        display(Image(filename=str(path)))
    else:
        print(f"(missing {fname} — run Part C first)")

## Conclusions

**225 init-probe runs + 225 full-pipeline runs** on tc07, tc09–tc12. Artifacts: `probe_results.json`, `pipeline_results.json`, `summary.json`, and σ-sweep plots in this folder.

### 1. Jitter does not help — σ=0 is optimal for every bound variant

Across all nine variant configs, the best mean final accuracy is always at **σ=0**. No bound variant gains more than 0 pp from noise; `p1_classic` picks up at most **+1.5 pp** at σ=0.2 (still near chance on tc07/tc09).

| variant | mean final @ σ=0 | mean final @ σ=0.5 | Δ σ=0.5 |
|---|---:|---:|---:|
| `p2_bound_on` | 0.906 | 0.685 | −0.221 |
| `p3_bound_on` | 0.890 | 0.635 | −0.255 |
| `p1_bound_on` | 0.880 | 0.622 | −0.258 |
| `p2_bound_off` | 0.857 | 0.624 | −0.233 |
| `p1_classic` | 0.503 | 0.503 | 0.000 |

### 2. Bound init degrades smoothly with σ; classic is already at the floor

- **Init probe (no finetune):** `bound_on` mean accuracy falls from **0.916 → 0.656** as σ goes 0 → 0.5. The decay is roughly linear in σ on tc09/tc10 (cardinality TCs).
- **Full pipeline:** same pattern — init accuracy at epoch 0 tracks the probe closely; protected finetune cannot recover a jittered init. At σ=0.5, bound variants land barely above vanilla (~0.64).
- **Classic init** stays at ~0.50 on tc07/tc09 regardless of σ. Jitter cannot inject the missing (relation ⊙ neighbour-class) signal; mild σ (0.05–0.2) does not break the protected freeze enough to help.

### 3. Direction-aware cardinality survives jitter

`bound_on` (rolled incoming code) beats `bound_off` (shared) at **every σ** on tc09–tc12. Example mean final accuracies:

- tc10: `p2_bound_on` 0.868 → 0.590 vs `p2_bound_off` 0.725 → 0.510 at σ=0 vs σ=0.5
- tc09: `p2_bound_on` 0.873 vs `p2_bound_off` 0.838 at σ=0 (same ~7 pp gap as `direction_aware_cardinality.ipynb`)

Noise erodes both tags proportionally; it does **not** close the directional gap.

### 4. P1 / P2 / P3 ordering is stable at low σ

At σ=0, mean final accuracy over the five TCs: **P2 ≥ P3 > P1** for bound (`p2_bound_on` 0.906, `p3_bound_on` 0.890, `p1_bound_on` 0.880). P3 does not beat P2 here — tc09/tc10 favour P2's direct-subclass codes. Jitter preserves the ordering until σ≥0.2, when everything collapses toward ~0.55–0.65.

### 5. Practical σ budget

| σ | mean Δ final (all bound) | comment |
|---|---:|---|
| 0.05 | −0.01 to −0.03 | small but consistently negative |
| 0.10 | −0.03 to −0.05 | noticeable on tc09/tc10 |
| 0.20 | −0.08 to −0.15 | init probe still >0.65 for P2 bound |
| 0.50 | −0.22 to −0.26 | destroys bound advantage |

**Takeaway:** Gaussian init jitter is not a useful augmentation for this recipe. The concept-bound vectors already encode the task features at σ=0; any additive noise is pure corruption. If regularization is desired, it should target finetune dynamics (LR, dropout in the probe, etc.) rather than perturbing the carefully constructed init.